# Ingest drivers.json file
1. Read the file using Spark DataFrame Reader API
1. Add Metadata Columns:
    - Source Files
    - Timestamp
1. Write to Bronze delta table

In [0]:
dbutils.widgets.text('p_batch_id', "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

#### Step 1: Read the json file using the dataframe reader API

In [0]:
#Define Schema (the drivers.json file has the 'name' object as a nested object i.e. given_name+family_name. Hence we use a nested schema for the 'name' object and another schema for the remaining objects)

from pyspark.sql.types import StructType, StructField, StringType,DateType

name_schema = StructType(fields=[
    StructField('givenName', StringType()),
    StructField('familyName', StringType())
])

drivers_schema = StructType(fields=[
    StructField('driverId', StringType()),
    StructField('name', name_schema),
    StructField('dateOfBirth', DateType()),
    StructField('nationality', StringType()),
    StructField('url', StringType())
])

In [0]:
#Read the drivers file

drivers_df = (
    spark.read
        .format('json')
        .schema(drivers_schema)
        .option('mode', 'FAILFAST')
        .load(source_file)
)

In [0]:
display(drivers_df)

driverId,name,dateOfBirth,nationality,url
abate,"List(carlo, abate)",1932-07-10,italian,http://en.wikipedia.org/wiki/Carlo_Mario_Abate
abecassis,"List(george, abecassis)",1913-03-21,british,http://en.wikipedia.org/wiki/George_Abecassis
acheson,"List(kenny, acheson)",1957-11-27,british,http://en.wikipedia.org/wiki/Kenny_Acheson
adams,"List(philippe, adams)",1969-11-19,belgian,http://en.wikipedia.org/wiki/Philippe_Adams
ader,"List(walt, ader)",1913-12-15,american,http://en.wikipedia.org/wiki/Walt_Ader
adolff,"List(kurt, adolff)",1921-11-05,german,http://en.wikipedia.org/wiki/Kurt_Adolff
agabashian,"List(fred, agabashian)",1913-08-21,american,http://en.wikipedia.org/wiki/Fred_Agabashian
ahrens,"List(kurt, ahrens)",1940-04-19,german,"http://en.wikipedia.org/wiki/Kurt_Ahrens,_Jr."
aitken,"List(jack, aitken)",1995-09-23,british,http://en.wikipedia.org/wiki/Jack_Aitken
albers,"List(christijan, albers)",1979-04-16,dutch,http://en.wikipedia.org/wiki/Christijan_Albers


#### Step 2: Add Metadata Columns
 - Source File
 - Ingestion Timestamp

In [0]:
drivers_final_df = add_ingestion_metadata(drivers_df)

####Step 3: Write to Bronze Delta Table

In [0]:
write_to_bronze(
    input_df = drivers_final_df,
    target_table = table_name,
    batch_id = v_batch_id
)

In [0]:
display(spark.table(table_name))

driverId,name,dateOfBirth,nationality,url,ingestion_timestamp,source_file,batch_id
abate,"List(carlo, abate)",1932-07-10,italian,http://en.wikipedia.org/wiki/Carlo_Mario_Abate,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
abecassis,"List(george, abecassis)",1913-03-21,british,http://en.wikipedia.org/wiki/George_Abecassis,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
acheson,"List(kenny, acheson)",1957-11-27,british,http://en.wikipedia.org/wiki/Kenny_Acheson,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
adams,"List(philippe, adams)",1969-11-19,belgian,http://en.wikipedia.org/wiki/Philippe_Adams,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
ader,"List(walt, ader)",1913-12-15,american,http://en.wikipedia.org/wiki/Walt_Ader,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
adolff,"List(kurt, adolff)",1921-11-05,german,http://en.wikipedia.org/wiki/Kurt_Adolff,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
agabashian,"List(fred, agabashian)",1913-08-21,american,http://en.wikipedia.org/wiki/Fred_Agabashian,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
ahrens,"List(kurt, ahrens)",1940-04-19,german,"http://en.wikipedia.org/wiki/Kurt_Ahrens,_Jr.",2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
aitken,"List(jack, aitken)",1995-09-23,british,http://en.wikipedia.org/wiki/Jack_Aitken,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
albers,"List(christijan, albers)",1979-04-16,dutch,http://en.wikipedia.org/wiki/Christijan_Albers,2026-08-02T16:47:29.780Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/drivers.json,2025-01
